<a href="https://colab.research.google.com/github/2303A51908/Reinforecement-Learning---B12/blob/main/2303A51908_RL_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# reinforce_cartpole.py
import numpy as np
np.bool8 = np.bool_

import gym
import time
import math
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Categorical

# ---------------------
# Hyperparameters
# ---------------------
ENV_ID = "CartPole-v1"
SEED = 1
GAMMA = 0.99
LR = 1e-3
HIDDEN_SIZE = 128
MAX_EPISODES = 1000
PRINT_INTERVAL = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BASELINE = False  # If True, implement a value network baseline (not implemented below)

# ---------------------
# Utilities for gym compatibility
# ---------------------
def safe_reset(env):
    res = env.reset()
    return res[0] if isinstance(res, tuple) else res

def safe_step(env, action):
    res = env.step(action)
    if len(res) == 5:  # gym >=0.26 (obs, reward, terminated, truncated, info)
        obs, reward, terminated, truncated, info = res
        done = terminated or truncated
        return obs, reward, done, info
    obs, reward, done, info = res
    return obs, reward, done, info

def set_seed(env, seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    env.seed(seed) if hasattr(env, 'seed') else None
    try:
        env.action_space.seed(seed)
    except:
        pass

# ---------------------
# Policy Network
# ---------------------
class PolicyNet(nn.Module):
    def __init__(self, obs_dim, action_dim, hidden=HIDDEN_SIZE):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim)
        )

    def forward(self, x):
        logits = self.net(x)
        return torch.softmax(logits, dim=-1)

# ---------------------
# Helper: compute discounted rewards-to-go
# ---------------------
def compute_returns(rewards, gamma=GAMMA):
    # reward-to-go: G_t = r_t + gamma*r_{t+1} + ...
    returns = []
    R = 0.0
    for r in reversed(rewards):
        R = r + gamma * R
        returns.insert(0, R)
    return returns

# ---------------------
# Training loop
# ---------------------
def train():
    env = gym.make(ENV_ID)
    set_seed(env, SEED)
    obs_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    policy = PolicyNet(obs_dim, action_dim).to(DEVICE)
    optimizer = torch.optim.Adam(policy.parameters(), lr=LR)

    running_rewards = []

    for episode in range(1, MAX_EPISODES + 1):
        obs = safe_reset(env)
        log_probs = []
        rewards = []
        done = False

        # generate one episode (full trajectory)
        while not done:
            obs_tensor = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            probs = policy(obs_tensor).squeeze(0)
            dist = Categorical(probs)
            action = dist.sample().item()
            log_prob = dist.log_prob(torch.tensor(action, device=DEVICE))
            next_obs, reward, done, _ = safe_step(env, action)

            log_probs.append(log_prob)
            rewards.append(reward)

            obs = next_obs

        # compute returns
        returns = compute_returns(rewards)
        returns = torch.tensor(returns, dtype=torch.float32, device=DEVICE)
        # optional: normalize returns for stability
        returns = (returns - returns.mean()) / (returns.std(unbiased=False) + 1e-8)

        # policy gradient step (REINFORCE)
        policy_loss = []
        for log_prob, G in zip(log_probs, returns):
            policy_loss.append(-log_prob * G)
        policy_loss = torch.stack(policy_loss).sum()

        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        episode_return = sum(rewards)
        running_rewards.append(episode_return)

        if episode % PRINT_INTERVAL == 0:
            avg_return = np.mean(running_rewards[-PRINT_INTERVAL:])
            print(f"Episode {episode}\tAvgReturn(last {PRINT_INTERVAL}) = {avg_return:.2f}")

    env.close()
    print("Training finished.")

if __name__ == "__main__":
    start = time.time()
    train()
    print("Elapsed:", time.time() - start)


/usr/local/lib/python3.12/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.12/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.12/dist-packages/gym/core.py:256: DeprecationWarning: WARN: Function `env.seed(seed)` is marked as deprecated and will be removed in the future. Please use `env.reset(seed=seed)` instead.
  deprecation(


Episode 10	AvgReturn(last 10) = 22.60
Episode 20	AvgReturn(last 10) = 21.10
Episode 30	AvgReturn(last 10) = 32.90
Episode 40	AvgReturn(last 10) = 30.30
Episode 50	AvgReturn(last 10) = 33.00
Episode 60	AvgReturn(last 10) = 44.50
Episode 70	AvgReturn(last 10) = 41.50
Episode 80	AvgReturn(last 10) = 52.10
Episode 90	AvgReturn(last 10) = 52.30
Episode 100	AvgReturn(last 10) = 68.90
Episode 110	AvgReturn(last 10) = 168.70
Episode 120	AvgReturn(last 10) = 168.70
Episode 130	AvgReturn(last 10) = 205.10
Episode 140	AvgReturn(last 10) = 170.40
Episode 150	AvgReturn(last 10) = 220.40
Episode 160	AvgReturn(last 10) = 228.00
Episode 170	AvgReturn(last 10) = 210.50
Episode 180	AvgReturn(last 10) = 207.00
Episode 190	AvgReturn(last 10) = 175.50
Episode 200	AvgReturn(last 10) = 167.20
Episode 210	AvgReturn(last 10) = 203.90
Episode 220	AvgReturn(last 10) = 216.10
Episode 230	AvgReturn(last 10) = 354.60
Episode 240	AvgReturn(last 10) = 261.60
Episode 250	AvgReturn(last 10) = 362.20
Episode 260	AvgRetu